# KavachX: run the three assessment demonstrations in Colab

**Use a standard CPU runtime, including Colab's latest Python 3.13 runtime.**
No paid GPU, API key, Drive mount or retraining is needed.
Run each cell in order. Upload **KavachX_Project_Share.zip** when prompted. This bundle contains all three
current task packages; cloning the original GitHub repository will not supply these repaired versions.

These are documented substitute demonstrations because the original assignment assets were unavailable.
Task 2 motion gating is still outstanding. Task 2 missing-helmet recall is weak; Task 3 previously achieved
64.7% precision and 73.3% recall, which does not justify automatic suppression. This notebook reports fresh
results and does not improve accuracy merely by moving execution to Colab.

The notebook kernel may use Python 3.13. Setup downloads a separate managed Python 3.12 interpreter
for the task scripts using pinned uv; it leaves the notebook kernel's Python and ML libraries unchanged.
Keep the notebook/data private. Initial setup downloads pinned packages and pretrained model weights.
Results are on temporary Colab storage until you download them in the final cell.


In [ ]:
from google.colab import files
from pathlib import Path
import datetime, hashlib, io, json, os, stat, subprocess, sys, zipfile

BASE = Path("/content") / ("kavachx_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
BASE.mkdir()
print("Working directory:", BASE, "\nPython:", sys.version)
uploaded = files.upload()
EXPECTED_HASHES = {'KavachX_Task1_Complete.zip': '19af00d9252f2fb0c908ebb8dd16028ef6c3c855f7eaef75cfb8503fa37ebb98', 'KavachX_Task2_Recovered.zip': '297a28dd6535c787312b3118fc0e3ecf6c9a9e71f92dce40ef87556ae01c56a3', 'KavachX_Task3_Complete.zip': '04a185501e6aea16df5154cfcbe1dc78186674eb7853bb2106dfb518b188831e'}

def unpack_checked(data, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        if archive.testzip() is not None:
            raise ValueError("Corrupt ZIP")
        for member in archive.infolist():
            path = (destination / member.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(destination)

if "KavachX_Project_Share.zip" in uploaded:
    data = uploaded["KavachX_Project_Share.zip"]
    unpack_checked(data, BASE / "project")
    T1 = T2 = T3 = BASE / "project/KavachX_Project"
    manifest_path = T1 / "FILE_HASHES.json"
    inventory = json.loads(manifest_path.read_text())
    for name, expected in inventory.items():
        path = (T1 / name).resolve()
        if not path.is_relative_to(T1.resolve()) or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError(f"Project file hash mismatch: {name}")
    EXPECTED_HASHES = {"project_inventory": hashlib.sha256(manifest_path.read_bytes()).hexdigest()}
    print("Verified consolidated project:", len(inventory), "files")
elif "KavachX_Colab_Run_All.zip" in uploaded:
    with zipfile.ZipFile(io.BytesIO(uploaded["KavachX_Colab_Run_All.zip"])) as outer:
        for number, (name, digest) in enumerate(EXPECTED_HASHES.items(), start=1):
            data = outer.read(name)
            if hashlib.sha256(data).hexdigest() != digest:
                raise ValueError(f"Package hash mismatch: {name}")
            unpack_checked(data, BASE / f"task{number}")
            print("Verified and extracted:", name)
    T1, T2, T3 = BASE / "task1", BASE / "task2", BASE / "task3/KavachX_Task3"
else:
    raise ValueError("Upload KavachX_Project_Share.zip")
RESULTS = BASE / "results"
RESULTS.mkdir()
for folder in (T1, T2, T3):
    if not (folder / "README.md").is_file():
        raise FileNotFoundError(folder / "README.md")
del uploaded, data


## Install in a separate environment
This leaves the notebook kernel's NumPy/Torch installation alone. All task scripts run as fresh subprocesses
in the isolated environment. CPU PyTorch is intentional. Linux ONNX timings will differ from the saved Mac results.


In [ ]:
VENV = BASE / "venv"
PY = VENV / "bin/python"
ENV = dict(os.environ, MPLBACKEND="Agg", MPLCONFIGDIR=str(BASE / "mplconfig"),
           CUDA_VISIBLE_DEVICES="", PYTHONUNBUFFERED="1")
command_number = 0
command_records = []
task1_audited = task2_benchmarked = task3_audited = False

def command(arguments, cwd=BASE, environment=None):
    global command_number
    command_number += 1
    log = RESULTS / f"command_{command_number:02d}.log"
    print("Running:", " ".join(map(str, arguments)), flush=True)
    with log.open("w") as file:
        file.write("Command: " + repr(list(map(str, arguments))) + "\n")
        with subprocess.Popen(list(map(str, arguments)), cwd=cwd, env=ENV if environment is None else environment,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                file.write(line)
            returncode = process.wait()
        file.write(f"\nExit code: {returncode}\n")
    command_records.append({"arguments": list(map(str, arguments)), "exit_code": returncode, "log": log.name})
    if returncode:
        raise RuntimeError(f"Command failed with exit code {returncode}. Read {log}")

def run(root, *arguments):
    command([PY, "-u", *arguments], root)

bootstrap = BASE / "bootstrap"
command([sys.executable, "-m", "pip", "install", "--target", bootstrap, "uv==0.9.5"])
uv_environment = dict(ENV, PYTHONPATH=str(bootstrap), UV_CACHE_DIR=str(BASE / "uv-cache"),
                      UV_PYTHON_INSTALL_DIR=str(BASE / "python"))
command([sys.executable, "-m", "uv", "venv", "--python", "3.12",
         "--python-preference", "only-managed", str(VENV)], environment=uv_environment)
run(BASE, "-c", "import sys; print('Task interpreter:',sys.version); assert sys.version_info[:2] == (3,12)")
PIP = [sys.executable, "-m", "pip", "--python", str(PY)]
command([*PIP, "install", "torch==2.8.0", "torchvision==0.23.0",
         "--index-url", "https://download.pytorch.org/whl/cpu"])
requirements = set()
for file in [T1 / "task1/requirements.txt", T2 / "requirements.txt", T3 / "task3/requirements.txt"]:
    for line in file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            # The source Mac archive records a newer matplotlib build. Use a known portable release.
            requirements.add("matplotlib==3.10.7" if line.startswith("matplotlib==") else line)
requirement_file = BASE / "requirements-colab.txt"
requirement_file.write_text("\n".join(sorted(requirements)) + "\n")
command([*PIP, "install", "-r", requirement_file])
command([*PIP, "check"])
command([*PIP, "freeze"])
run(T1, "-c", "import torch,numpy,onnxruntime,platform; print(platform.platform()); "
    "print('Torch:',torch.__version__,'NumPy:',numpy.__version__,'ORT:',onnxruntime.__version__); "
    "print('CUDA available:',torch.cuda.is_available())")
print("Setup complete. No kernel restart is needed: task execution uses the separate interpreter.")


## Task 1: English rule → tracked dwell event → masked JSON evidence
Run the 35-second bag demonstration, a five-second negative control, and a car rule using the same model.
The source video is not a loading bay. The saved bag event was borderline and required verification.


In [ ]:
RUN1 = RESULTS / "task1"
run(T1, "task1/prepare_demo.py")
run(T1, "-m", "unittest", "discover", "-s", "tests", "-p", "test_task1.py", "-v")
run(T1, "task1/run_task1.py", "--model", "models/yolov8s-worldv2.pt",
    "--output-dir", RUN1 / "mot17_bag_demo")
run(T1, "task1/run_task1.py", "--model", "models/yolov8s-worldv2.pt", "--max-frames", "150",
    "--output-dir", RUN1 / "short_clip_negative")
run(T1, "task1/run_task1.py", "--model", "models/yolov8s-worldv2.pt", "--max-frames", "150",
    "--rule", "Alert me if a car enters the parking and stays more than 3 seconds.",
    "--output-dir", RUN1 / "car_noun_smoke")
run(T1, "task1/verify_demo.py", "--runs-root", RUN1)
task1_audited = True


## Task 2: evaluate the trained checkpoint, export and benchmark
Reuses the recovered 15-epoch checkpoint and 240/60 public PPE subset. **No retraining occurs here.**
FP32, FP16 and convolution-only INT8 PTQ are evaluated with consistent preprocessing. The original source
labels have known quality problems. The motion-gating experiment remains separate and is not run or claimed completed.


In [ ]:
RUN2 = RESULTS / "task2"
run(T2, "-c", "from pathlib import Path; import yaml; p=Path('data/ppe/data.yaml'); "
    "d=yaml.safe_load(p.read_text()); d['path']=str(p.parent.resolve()); "
    "p.write_text(yaml.safe_dump(d,sort_keys=False))")
run(T2, "-m", "unittest", "discover", "-s", "tests", "-p", "test_task2.py", "-v")
run(T2, "task2/evaluate.py", "--output-dir", RUN2)
run(T2, "task2/quantize_pipeline.py", "--output-dir", RUN2)
run(T2, "task2/benchmark.py", "--output-dir", RUN2)
task2_benchmarked = True
print((RUN2 / "benchmark_table.md").read_text())


## Task 3: score snapshots, calibrate threshold, evaluate 30 held-out images
Uses original BIRDSAI thermal images with human-presence proxy labels, not verified intrusion/authorization labels.
There are 20 calibration images and 30 test images. This re-evaluates the same fixed test set; it is not a new unseen test.
The script saves held alerts for audit. A result with missed positives must not be presented as safe automatic suppression.


In [ ]:
RUN3 = RESULTS / "task3"
run(T3, "task3/prepare_demo.py")
run(T3, "-m", "unittest", "discover", "-s", "tests", "-p", "test_task3.py", "-v")
run(T3, "task3/verifier.py", "--output-dir", RUN3)
run(T3, "task3/evaluate.py", "--scores", RUN3 / "verifier_scores.csv", "--output-dir", RUN3)
run(T3, "task3/verify_demo.py", "--run-dir", RUN3, "--rescore")
task3_audited = True
print((RUN3 / "verifier_evaluation.md").read_text())


## View fresh results and download
This saves new run artifacts and logs, not a combined submission or a replacement for the source packages.
Keep the download: Colab runtime files are temporary. If any earlier cell fails, the download cell can still
save available logs; the archive is marked partial. Do not confuse partial execution with a passed run.


In [ ]:
import shutil
from IPython.display import display, Image as DisplayImage

for folder in [RESULTS / "task1/mot17_bag_demo", RESULTS / "task2"]:
    images = sorted(folder.glob("masked_frame*.jpg"))[:2] + sorted(folder.glob("failure_*_actual.jpg"))[:2]
    for path in images:
        print(path.name)
        display(DisplayImage(filename=str(path), width=640))
required = [RESULTS / "task1/mot17_bag_demo/sample_event.json",
            RESULTS / "task2/benchmark_table.csv", RESULTS / "task3/evaluation.json"]
completed = (all(path.is_file() for path in required) and
             all(globals().get(flag, False) for flag in ("task1_audited", "task2_benchmarked", "task3_audited")) and
             not any(record["exit_code"] for record in globals().get("command_records", [])))
(RESULTS / "RUN_STATUS.json").write_text(json.dumps({
    "execution": "All three demonstration runs and checks finished" if completed else "PARTIAL: inspect command logs",
    "commands": globals().get("command_records", []),
    "source_packages_sha256": EXPECTED_HASHES,
    "limitations": ["Substitute assets; original assessment media not supplied",
                    "Task 2 motion gating not completed by this notebook",
                    "PPE accuracy and verifier recall require further work",
                    "Presence proxy labels do not establish unauthorized intrusion"]}, indent=2))
if (BASE / "requirements-colab.txt").is_file():
    shutil.copy2(BASE / "requirements-colab.txt", RESULTS / "requirements-colab.txt")
output_zip = shutil.make_archive(str(BASE / "KavachX_Colab_Results"), "zip", RESULTS)
print("Download:", output_zip)
files.download(output_zip)
